# Notebook 01 — TwinVerse dataset audit and provenance

This notebook inspects the source material currently tracked by the Rounds backend, relates it to the challenge uploads, and creates a reproducible audit report. It uses the existing sponsor-provided profiles, faculty CSV, rubric, clinician personas, and project cases.

**This is a structural and provenance audit.** It does not certify clinical accuracy, label student performance, or measure learning. Those need faculty review and student evidence. The external dialogue and pathology dataset links are listed but are not cached in this repository.

## Questions this stage answers

1. What data is actually available in the repository, and where did it come from?
2. Can it be parsed reliably and mapped to the project schema?
3. Which sources can support case/environment generation, action extraction, rubric scoring, or later learning evaluation?
4. What is missing before training an evaluator or making a learning-efficiency claim?

The audit keeps event-provided versions separate. If the faculty CSV and the richer profile collection describe the same named person differently, that is a review item; the importer must not silently combine them.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "PRD.md").exists():
    REPO_ROOT = REPO_ROOT.parent
BACKEND = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND))

from scripts.audit_datasets import build_report

print("Repository:", REPO_ROOT)
print("Notebook requirements: backend/requirements-notebooks.txt")

## Run the same audit used by the command-line script

The Python module is the source of truth; this notebook visualizes its output. That keeps the notebook and the end-to-end script from drifting apart.

In [ ]:
report = build_report(REPO_ROOT)
report_path = REPO_ROOT / "docs/validation/dataset_audit.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print(f"Gate: {report['gate']}")
print(f"Sources inventoried: {report['source_count']}")
print(f"Errors: {len(report['errors'])}; review notes: {len(report['warnings'])}")
for item in report["errors"]:
    print("ERROR:", item)
for item in report["warnings"]:
    print("REVIEW:", item)

## Source inventory

Counts describe files available to this project. An external link with no cached file is shown as unavailable for local training. SHA-256 hashes make changes to local inputs visible in subsequent audits.

In [ ]:
sources = pd.DataFrame(report["sources"])
view_cols = [c for c in ["id", "kind", "local_status", "record_count", "license", "sha256"] if c in sources]
sources[view_cols]

In [ ]:
plot_sources = sources.copy()
plot_sources["count_for_plot"] = pd.to_numeric(plot_sources["record_count"], errors="coerce").fillna(0)
fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ["#2563eb" if status == "present" else "#9ca3af" for status in plot_sources["local_status"]]
ax.barh(plot_sources["id"], plot_sources["count_for_plot"], color=colors)
ax.set_xlabel("Records / rubric domains (external uncached sources show as 0)")
ax.set_title("What is locally auditable in this repository?")
ax.grid(axis="x", alpha=.2)
plt.tight_layout()
plt.show()

## Faculty patient seed table

The source has eight patient rows and ten fields. We inspect only a small subset here. Names and scenarios are the event-provided synthetic examples; they are not real patient records or learner outcomes.

In [ ]:
patient_csv = REPO_ROOT / "backend/app/data/source_fixtures/faculty-provided-sample-patient-details.csv"
patients = pd.read_csv(patient_csv)
print("Rows:", len(patients), "Columns:", len(patients.columns))
display(patients[["patient-name", "age", "background", "personality"]].head(8))

## Richer profile records and version comparison

The VHS sample file contains eight adjacent pretty-printed JSON objects rather than a single JSON array or conventional JSON Lines. The existing importer handles that format. We compare names and ages against the CSV, but name matches alone do not mean that two case versions should be merged.

In [ ]:
from app.studio.vhs_import import load_profiles

vhs_path = REPO_ROOT / "backend/app/data/vhs/virtual-patient-profiles-sample.json"
vhs_profiles = load_profiles(vhs_path)
profile_rows = [
    {"name": p.get("patient_overview", {}).get("name"),
     "age": p.get("patient_overview", {}).get("age"),
     "learning_opportunities": len(p.get("learning_opportunities", {}).get("opportunities", [])),
     "disclosure_tiers": sum(len(v) for v in p.get("information_disclosure", {}).values() if isinstance(v, list))}
    for p in vhs_profiles
]
profile_df = pd.DataFrame(profile_rows)
display(profile_df)
print("Name matches across CSV and VHS:", report["cross_source_comparison"])

### Run the existing deterministic VHS-to-case normalizer

The repository importer converts profile facts into draft case objects and preserves source paths. The preview below shows normalized case size. This is a schema/import check, not faculty approval; cases remain drafts until reviewed.


In [ ]:
from app.studio.vhs_import import import_all

normalized_cases = import_all(vhs_path)
normalized_preview = pd.DataFrame([
    {"case_id": case.case_id, "patient": case.profile.name, "draft": all(f.status == "draft" for f in case.facts),
     "facts": len(case.facts), "labs": len(case.labs), "objectives": len(case.objectives),
     "with_source": sum(bool(f.source.ref) for f in case.facts)}
    for case in normalized_cases
])
display(normalized_preview)
print("Normalized draft cases:", len(normalized_cases))


In [ ]:
vhs_audit = next(s for s in report["sources"] if s["id"] == "vhs_virtual_patient_profiles")
tier_counts = vhs_audit["disclosure_fact_count_by_tier"]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(list(tier_counts), list(tier_counts.values()), color=["#60a5fa", "#34d399", "#fbbf24", "#f87171"])
ax.set_ylabel("Profile facts listed at each disclosure tier")
ax.set_title("Disclosure structure in the eight supplied profiles")
ax.grid(axis="y", alpha=.2)
plt.tight_layout()
plt.show()

## Rubric structure

The SHP encounter rubric has seven domains, ratings 1–4 or N/A, and a maximum total of 28. A score of 1 in domains 6 or 7 triggers review. The rubric says the total is a discussion aid, not a pass/fail grade. It supplies assessment descriptors, **not** transcripts with ground-truth scores, and does not define weighted leaf criteria.

In [ ]:
rubric_path = REPO_ROOT / "backend/app/data/rubric_shp_v1.json"
rubric = json.loads(rubric_path.read_text(encoding="utf-8"))
domains = pd.DataFrame([{"id": d["id"], "domain": d["name"],
                         "review_on_1": d["triggers_review_if_score_1"],
                         "has_1_to_4_descriptors": all(str(s) in d["descriptors"] for s in range(1, 5))}
                        for d in rubric["domains"]])
display(domains)

# A completeness view, not an invented score distribution.
complete = [[int(str(score) in d["descriptors"]) for score in range(1, 5)] for d in rubric["domains"]]
fig, ax = plt.subplots(figsize=(7, 4.5))
image = ax.imshow(complete, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(4), ["1", "2", "3", "4"])
ax.set_yticks(range(len(domains)), domains["domain"])
ax.set_xlabel("Rubric rating")
ax.set_title("Descriptor availability by rubric domain")
for row in range(len(complete)):
    for col in range(4):
        ax.text(col, row, "defined" if complete[row][col] else "missing", ha="center", va="center", fontsize=8)
plt.tight_layout()
plt.show()

### Specialty rubric available for later case mapping

The supplied pediatric aural-rehabilitation rubric has four point categories: introduction/professionalism (10), needs assessment (30), audiogram review (30), and realistic rehabilitation expectations (30). The document totals 100 points. It is a separate specialty rubric, not a replacement for the SHP encounter rubric. The local bundle has no matching case, audiogram, or Speech Banana image, so those rubric opportunities cannot yet be exercised faithfully.


In [ ]:
specialty = next(s for s in report["sources"] if s["id"] == "pediatric_aural_rehabilitation_rubric")
pd.DataFrame({"Category": ["Introduction and professionalism", "Needs assessment", "Audiogram review", "Aural rehab expectations"],
              "Points": specialty["point_values"]})


## Dataset-to-model readiness map

This is a qualitative inventory of what each source could support, not a model-performance result. `Candidate` means additional labels, license checks, and review are needed. None of the current sources is ground truth for student learning gains.

In [ ]:
readiness = pd.DataFrame([
    ("Faculty patient CSV", "Present", "Case fields", "No student labels"),
    ("VHS patient profiles", "Present", "Disclosure and objectives", "No student labels"),
    ("Clinician personas", "Present", "Role-play scenarios", "Not student outcomes"),
    ("SHP rubric", "Present", "Scoring targets", "No scored encounters"),
    ("Patient–Doctor dialogue", "External link only", "Candidate action examples", "No validated student scores"),
    ("Pathology results", "External link only", "Candidate lab exercises", "Not learning outcomes"),
    ("Project cases", "Present", "Environment tests", "Not learner evidence"),
], columns=["Source", "Availability", "Potential use", "Cannot establish"])
display(readiness)

## Gate decision

The PASS gate covers source availability, expected counts, parseability, and the rubric/case structure. Review notes remain visible: external sources are not cached, versions need faculty review, and no student outcomes exist. So this permits **Notebook 02 data preparation**, while supervised action-model training still needs a reviewed annotation set or suitable licensed labeled corpus.

In [ ]:
print("Gate:", report["gate"])
print("Next:", report["advance_to"])
print("Checks:")
for check, passed in report["checks"].items():
    print(f"  {'PASS' if passed else 'NOT EVALUABLE'}  {check}")
print("\nConstraints for the next stage:")
for item in report["next_stage_constraints"]:
    print("-", item)
print("\nAudit report:", report_path)